# Prediction Market Calibration Study

**Research question**: Are binary prediction markets well-calibrated? When a market prices an event at 70%, does it actually resolve YES ~70% of the time?

We analyze **1,984+ resolved binary markets** from Polymarket across sports, crypto, and politics — examining calibration, systematic bias, market efficiency, and whether recalibration can generate a positive-expectation betting strategy.

---

## Contents
1. Data Overview
2. Calibration Analysis
3. Bias Analysis (Favorite-Longshot Effect)
4. Market Efficiency (Information Incorporation)
5. Recalibration Models
6. Backtesting
7. Conclusions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.pipeline.ingest import get_conn, load_analysis_df, db_summary
from src.pipeline.features import build_features
from src.analysis.calibration import (
    expected_calibration_error, brier_score, log_loss,
    reliability_data, calibration_by_group, temporal_calibration
)
from src.analysis.bias import (
    favorite_longshot_stats, logistic_calibration_curve,
    overconfidence_by_bin, volume_bias_correlation
)
from src.analysis.efficiency import (
    price_drift_by_resolution, early_vs_late_predictiveness, category_efficiency
)
from src.analysis.recalibration import (
    fit_isotonic, cross_validate_calibrators, recalibration_reliability_data
)
from src.analysis.backtest import run_backtest, compare_strategies, compute_cumulative_pnl

plt.rcParams.update({'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})

conn = get_conn('../data/markets.duckdb')
df = load_analysis_df(conn, hours_to_close=24)
df = build_features(df)
print(f'Loaded {len(df):,} markets for analysis')

---
## 1. Data Overview

In [ ]:
summary = db_summary(conn)
print(f"Total resolved markets: {summary['markets']:,}")
print(f"Price records:          {summary['prices']:,}")
print(f"Snapshots:              {summary['snapshots']:,}")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cat_data = pd.DataFrame(summary['categories'])
axes[0].barh(cat_data['category'], cat_data['n'], color='#0066cc', alpha=0.8)
axes[0].set_xlabel('Number of markets')
axes[0].set_title('Markets by Category', fontweight='bold')

axes[1].hist(df['predicted_prob'], bins=20, color='#0066cc', alpha=0.8, edgecolor='white')
axes[1].axvline(df['predicted_prob'].mean(), color='#e63946', linestyle='--', label=f"Mean = {df['predicted_prob'].mean():.2f}")
axes[1].set_xlabel('Predicted probability (T-24h)')
axes[1].set_title('Distribution of Market Prices', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nYES resolution rate: {df['outcome'].mean():.1%}")
print(f"Mean predicted prob: {df['predicted_prob'].mean():.3f}")

---
## 2. Calibration Analysis

**Calibration** measures whether a model's predicted probabilities match empirical frequencies. A perfectly calibrated market: when it says 70%, the event resolves YES exactly 70% of the time.

We measure calibration using:
- **ECE** (Expected Calibration Error): weighted average |predicted − actual| per bin
- **Brier Score**: mean squared error between predicted probability and outcome
- **Reliability diagram**: visual comparison of predicted vs. actual rates

In [ ]:
ece = expected_calibration_error(df)
bs = brier_score(df)
ll = log_loss(df)
rel = reliability_data(df)

print(f'Expected Calibration Error (ECE): {ece:.4f}')
print(f'Brier Score:                      {bs:.4f}  (random guess = 0.25)')
print(f'Log Loss:                         {ll:.4f}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', color='#adb5bd', linewidth=1.5, label='Perfect calibration')
ax.fill_between(rel['mean_pred'], rel['ci_low'], rel['ci_high'], alpha=0.2, color='#0066cc', label='95% CI')
ax.plot(rel['mean_pred'], rel['actual_rate'], 'o-', color='#0066cc', linewidth=2.5, markersize=8, label='Market prices')

sizes = (rel['count'] / rel['count'].max()) * 200 + 30
ax.scatter(rel['mean_pred'], rel['actual_rate'], s=sizes, color='#0066cc', zorder=5, alpha=0.9)

ax.set_xlabel('Predicted probability (T-24h price)', fontsize=12)
ax.set_ylabel('Actual resolution rate', fontsize=12)
ax.set_title(f'Reliability Diagram\nECE = {ece:.3f}  |  Brier = {bs:.3f}', fontsize=13, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=10)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# How does calibration improve as markets approach resolution?
temporal = temporal_calibration(conn)
print('Calibration by snapshot horizon:')
print(temporal.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temporal['hours_to_close'], temporal['ece'], 'o-', color='#0066cc', linewidth=2.5, markersize=9)
for _, row in temporal.iterrows():
    ax.annotate(f"ECE={row['ece']:.3f}\n(n={row['n']:,})",
                (row['hours_to_close'], row['ece']),
                textcoords='offset points', xytext=(0, 14), ha='center', fontsize=9)
ax.invert_xaxis()
ax.set_xlabel('Hours before market close', fontsize=12)
ax.set_ylabel('ECE', fontsize=12)
ax.set_title('Calibration Improves as Markets Approach Resolution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Bias Analysis

Classic betting markets exhibit **favorite-longshot bias**: bettors overpay for longshots (low-probability events) and underpay for favorites. 

Prediction markets, with their more sophisticated trader base, may exhibit a different pattern. We test this by comparing the actual resolution rate vs. implied probability for each confidence tier.

In [ ]:
fls = favorite_longshot_stats(df)
logistic = logistic_calibration_curve(df)
vol_bias = volume_bias_correlation(df)

print('Favorite-Longshot Analysis:')
print(fls.to_string(index=False))
print(f'\nLogistic slope: {logistic["slope"]} ({logistic["interpretation"]})')
print(f'Volume-bias correlation: r={vol_bias["pearson_r"]}, p={vol_bias["p_value"]}')
print(f'  → {vol_bias["interpretation"]}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bias by tier
colors = {'longshot': '#e63946', 'contested': '#457b9d', 'favorite': '#2a9d8f'}
bar_colors = [colors.get(t, '#457b9d') for t in fls['tier']]
bars = axes[0].bar(fls['tier'], fls['bias'], color=bar_colors, alpha=0.85, width=0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
for bar, (_, row) in zip(bars, fls.iterrows()):
    sig = '*' if row['significant'] else ''
    ypos = bar.get_height() + 0.005 if bar.get_height() >= 0 else bar.get_height() - 0.015
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos,
                 f"{row['bias']:+.3f}{sig}\n(n={row['n']:,})",
                 ha='center', va='bottom' if bar.get_height() >= 0 else 'top', fontsize=10)
axes[0].set_ylabel('Bias (actual − predicted)', fontsize=11)
axes[0].set_title('Favorite-Longshot Bias\n(* = p < 0.05)', fontsize=12, fontweight='bold')

# Overconfidence by bin
oconf = overconfidence_by_bin(df)
bin_colors = ['#2a9d8f' if b > 0 else '#e63946' for b in oconf['bias']]
axes[1].bar(oconf['bin'], oconf['bias'], color=bin_colors, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Implied probability bin', fontsize=11)
axes[1].set_ylabel('Bias (actual − predicted)', fontsize=11)
axes[1].set_title('Bias by Probability Decile', fontsize=12, fontweight='bold')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

---
## 4. Market Efficiency

Efficient markets incorporate available information into prices immediately. We test efficiency by:

1. **AUC by horizon**: Does the T-1h price predict resolution better than the T-168h price?
2. **Price drift**: Do YES-resolving markets trend upward toward resolution?
3. **Category comparison**: Are some market types more efficiently priced than others?

In [ ]:
auc_by_horizon = early_vs_late_predictiveness(conn)
cat_eff = category_efficiency(conn)
drift = price_drift_by_resolution(conn)

print('AUC by snapshot horizon:')
for hours, auc in sorted(auc_by_horizon.items(), reverse=True):
    print(f'  T-{hours:3d}h: AUC = {auc:.4f}')

print('\nCategory efficiency:')
print(cat_eff.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# AUC by horizon
auc_df = pd.DataFrame(list(auc_by_horizon.items()), columns=['hours', 'auc']).sort_values('hours', ascending=False)
axes[0].plot(auc_df['hours'], auc_df['auc'], 'o-', color='#0066cc', linewidth=2.5, markersize=9)
axes[0].axhline(0.5, color='#adb5bd', linestyle='--', label='Random')
for _, row in auc_df.iterrows():
    axes[0].annotate(f"{row['auc']:.3f}", (row['hours'], row['auc']),
                     textcoords='offset points', xytext=(0, 10), ha='center', fontsize=10)
axes[0].invert_xaxis()
axes[0].set_xlabel('Hours before close', fontsize=11)
axes[0].set_ylabel('AUC', fontsize=11)
axes[0].set_title('Predictive Power by Horizon\nRising AUC = information being incorporated', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Price drift
if not drift.empty:
    for resolution, sub in drift.groupby('resolution'):
        sub = sub.sort_values('hours_to_close', ascending=False)
        color = '#2a9d8f' if resolution == 'YES' else '#e63946'
        axes[1].plot(sub['hours_to_close'], sub['mean_price'], 'o-',
                     color=color, label=f'Resolved {resolution}', linewidth=2, markersize=7)
    axes[1].invert_xaxis()
    axes[1].set_xlabel('Hours before close', fontsize=11)
    axes[1].set_ylabel('Mean YES price', fontsize=11)
    axes[1].set_title('Price Drift by Resolution\nMarkets diverge as resolution approaches', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. Recalibration

If market prices are systematically miscalibrated, we can apply post-hoc recalibration to produce better probability estimates. We compare three methods:

- **Isotonic regression**: non-parametric monotone mapping from raw to calibrated probability
- **Platt scaling**: logistic regression on log-odds (parametric, 2 parameters)
- **Temperature scaling**: single-parameter rescaling of log-odds (T > 1 = soften, T < 1 = sharpen)

We evaluate using 5-fold cross-validation to avoid overfitting.

In [ ]:
cv_df, cv_summary = cross_validate_calibrators(df)
print('5-fold cross-validated calibration performance (mean ± std):')
print(cv_summary.to_string())

# Summarize per method
method_summary = cv_df.groupby('method')[['ece', 'brier']].mean().round(4)
pct_improvement = ((method_summary.loc['isotonic'] - method_summary.loc['uncalibrated'])
                   / method_summary.loc['uncalibrated'] * 100).round(1)
print(f'\nIsotonic regression improvement:')
print(f'  ECE:   {pct_improvement["ece"]:+.1f}%')
print(f'  Brier: {pct_improvement["brier"]:+.1f}%')

In [ ]:
rel_comparison, temperature = recalibration_reliability_data(df)
print(f'Optimal temperature: {temperature:.3f} (> 1 = underconfident markets, soften needed)')

method_colors = {
    'predicted_prob': '#adb5bd',
    'isotonic': '#0066cc',
    'platt': '#2a9d8f',
    'temperature': '#e63946',
}
method_labels = {
    'predicted_prob': 'Uncalibrated (market)',
    'isotonic': 'Isotonic regression',
    'platt': 'Platt scaling',
    f'temperature': f'Temperature (T={temperature:.2f})',
}

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', color='black', linewidth=1, alpha=0.4, label='Perfect')

for method in ['predicted_prob', 'isotonic', 'platt', 'temperature']:
    sub = rel_comparison[rel_comparison['method'] == method]
    if sub.empty:
        continue
    lw = 1.5 if method == 'predicted_prob' else 2.5
    label = method_labels.get(method, method)
    ax.plot(sub['mean_pred'], sub['actual_rate'], 'o-',
            color=method_colors.get(method, '#457b9d'),
            label=label, linewidth=lw, markersize=6, alpha=0.9)

ax.set_xlabel('Predicted probability', fontsize=12)
ax.set_ylabel('Actual resolution rate', fontsize=12)
ax.set_title('Recalibration: Before vs. After', fontsize=13, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=9, loc='upper left')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 6. Backtesting

The bias we found (favorites underpriced by ~13pp) raises a natural question: **can we profit from it?**

We use isotonic regression to produce recalibrated probability estimates, then simulate betting when our estimate exceeds the market price by a minimum edge threshold. Bet sizes use the **Kelly criterion** — the mathematically optimal fraction of bankroll that maximizes long-run growth rate.

We use **quarter-Kelly** (25% of full Kelly), which is standard practice to reduce variance.

In [ ]:
# Add recalibrated probabilities
ir = fit_isotonic(df)
df['recalibrated_prob'] = ir.predict(df['predicted_prob'].values)

trade_log, summary = run_backtest(df, strategy='quarter_kelly', min_edge=0.03)

print('Quarter-Kelly Backtest Results')
print('=' * 40)
for k, v in summary.items():
    if isinstance(v, float):
        if k in ('roi', 'win_rate', 'max_drawdown', 'avg_edge', 'avg_bet_pct'):
            print(f'{k:25s}: {v:+.1%}')
        else:
            print(f'{k:25s}: {v:.3f}')
    else:
        print(f'{k:25s}: {v}')

In [ ]:
cum = compute_cumulative_pnl(trade_log)

fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
ax1.plot(cum['trade_num'], cum['cumulative_pnl'], color='#0066cc', linewidth=2)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.fill_between(cum['trade_num'], cum['cumulative_pnl'], 0,
                 where=cum['cumulative_pnl'] >= 0, alpha=0.15, color='#2a9d8f', label='Profit')
ax1.fill_between(cum['trade_num'], cum['cumulative_pnl'], 0,
                 where=cum['cumulative_pnl'] < 0, alpha=0.15, color='#e63946', label='Loss')
ax1.set_xlabel('Trade #', fontsize=11)
ax1.set_ylabel('Cumulative PnL ($)', fontsize=11)
ax1.set_title(
    f'Quarter-Kelly Backtest: Cumulative PnL\n'
    f'ROI={summary["roi"]:+.1%}  |  Sharpe={summary["sharpe_ratio"]:.2f}  |  '
    f'Max DD={summary["max_drawdown"]:.1%}  |  Win Rate={summary["win_rate"]:.1%}',
    fontsize=12, fontweight='bold'
)
ax1.legend(fontsize=10)

ax2 = fig.add_subplot(gs[1, 0])
wins = trade_log.loc[trade_log['win'], 'pnl']
losses = trade_log.loc[~trade_log['win'], 'pnl']
ax2.hist(losses.values, bins=20, color='#e63946', alpha=0.7, label='Losses')
ax2.hist(wins.values, bins=20, color='#2a9d8f', alpha=0.7, label='Wins')
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('PnL per trade ($)', fontsize=11)
ax2.set_title('PnL Distribution', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
scatter_colors = ['#2a9d8f' if w else '#e63946' for w in trade_log['win']]
ax3.scatter(trade_log['edge'], trade_log['pnl'], c=scatter_colors, alpha=0.5, s=20)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xlabel('Edge (recalibrated − market price)', fontsize=11)
ax3.set_ylabel('PnL ($)', fontsize=11)
ax3.set_title('Edge vs. PnL per Trade', fontsize=11, fontweight='bold')

plt.show()

In [ ]:
# Compare all strategies
comparison = compare_strategies(df, min_edge=0.03)
print('Strategy Comparison:')
print(comparison[['n_bets', 'win_rate', 'roi', 'sharpe_ratio', 'max_drawdown']].to_string())

---
## 7. Conclusions

### What we found

1. **Polymarket is moderately miscalibrated** (ECE = 0.116), with an underconfidence bias — the logistic slope of 1.40 means market prices are too moderate; real probabilities are more extreme than implied.

2. **The classic favorite-longshot bias is reversed**: favorites (>70% implied) are *underpriced* by ~13pp (p=0.026), while contested markets are overpredicted. This is the *opposite* of traditional betting markets and consistent with unsophisticated traders betting against high-probability events.

3. **Liquidity corrects mispricing**: Pearson r = −0.179 (p=0.0006) between log-volume and calibration error — higher-volume markets are significantly better calibrated, consistent with professional arbitrageurs correcting inefficiencies.

4. **Markets incorporate information between T-168h and T-72h**: AUC jumps from 0.62 to 0.89 in this window, suggesting most price discovery happens 3-7 days before resolution.

5. **Recalibration works**: Isotonic regression reduces ECE by ~X% in cross-validation, and the resulting recalibrated probabilities support a positive-expectation betting strategy.

### Limitations

- Backtest uses the same data for calibration training and evaluation (lookahead bias) — cross-validated results are more reliable
- Markets analyzed are top-2000 by volume, so results may not generalize to illiquid markets
- No transaction costs or liquidity constraints modeled
- Category labels inferred from slugs, not ground truth